# GridNet-HD — Robustness protocol กับ pose error แบบ "ร่วม" (shared): ทุกภาพของโซนผิดเหมือนกัน (GPU)

**ทำไมต้องมี**: sweep หลักของ paper ให้ทุกกล้องมี error ของตัวเองแบบสุ่มอิสระ (i.i.d.) ซึ่งเป็นกรณีที่ง่ายสำหรับโมเดลที่เฉลี่ยหลาย view
error จริงมักเอียงไปทางเดียวกันทุกภาพ notebook นี้รัน protocol เดิมทุกอย่าง (โหมด exact, 10 draws, ระดับ error ชุดเดิม) แต่เปลี่ยนอย่างเดียวคือ
**ทุกกล้องของโซนใช้ error ตัวเดียวกัน** (สุ่มหนึ่งครั้งต่อโซนต่อ draw จาก distribution เดียวกับ sweep i.i.d.):

| sweep | ความหมาย | ระดับ |
|---|---|---|
| `trans_shared` | เลื่อนกล้องทุกตัวด้วยเวกเตอร์เดียวกัน = ชุดภาพทั้งชุดเหลื่อมกับ point cloud (georeference สองชุดข้อมูลแยกกัน) | σ_t 2, 5, 10, 25, 50 cm |
| `rot_shared` | หมุนกล้องทุกตัวด้วยมุมเดียวกันรอบแกนของกล้อง = boresight error | σ_r 0.05, 0.1, 0.25, 0.5, 1° |

ไม่มีการเทรนอะไร: ใช้ head 24 ตัวของตาราง (4 แบบ × เทรนด้วย pose ที่ให้มา / pose-noise augmentation × 3 seeds) + LiDAR-only + ImageVote
ผลจะถูกวางเทียบกับ run `exact_v1` (error แบบ i.i.d. ขนาดเดียวกัน) ใน cell 7

- ต้องมีบน Drive แล้ว (มีครบจาก lab ก่อน ๆ): LiDAR cache + zone tar ของ val zones, feature bank (`GridNet/protocol/subsets_<RUN2D>/<zone>/subset.npz`),
  `GridNet/fusion/<head>/best.pth` ของ heads ใน `HEADS`, และ `GridNet/protocol/exact_v1/summary.json`
- **Runtime: GPU A100** (เครื่องที่โหมด exact เคยรันจริง)
- **ก่อนรัน**: `GridNet/code/gridnet_code.zip` บน Drive ต้องเป็นของ **update41 ขึ้นไป** (cell 4 ตรวจให้)
- เวลาที่วัดได้ (run `shared_v1`, 14 พ.ย. 2026, A100-SXM4-40GB): cell 6 ใช้ **139.5 นาที** = 16–28 นาทีต่อโซน, 13.7 วินาทีต่อ evaluation
  (612 evaluations × 26 โมเดล); เตรียมข้อมูล 6.5 นาที; ทั้ง session ราว 2.5 ชม. (ที่ประมาณไว้ก่อนรันคือ 1.9 ชม. สำหรับ cell 6)
  cell 2 พิมพ์ค่าประมาณตามค่าที่ตั้ง โดย interpolate ระหว่าง run ที่วัดแล้ว (6 โมเดล ~6 วินาที, 26 โมเดล 13.7, 37 โมเดล 14.4 วินาทีต่อ evaluation)
- **ผลลัพธ์**: `GridNet/protocol/<TAG>/` (results.json(.gz), summary.json, `compare_exact_v1.json/.txt`, รูป) — เสร็จแล้วผลอยู่บน Drive

**Resume**: Run all ซ้ำได้ — โซนที่เสร็จแล้วถูกข้าม แต่ถ้าเปลี่ยน `HEADS` / `SWEEPS` / ระดับ / `DRAWS` / `MODE` ให้ตั้ง `TAG` ใหม่ (script หยุดเองถ้าไม่ตรง)

In [ ]:
#@title 1. GPU (the A100 is the configuration the exact mode has run on; nothing is trained here)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.get_device_capability()[0] >= 8, "need an L4 or A100 runtime (the environment tarball is built for these GPUs)"


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
TAG = 'shared_v1'                                #@param {type:"string"}
# the 24 heads of the fix table: 4 designs x (delivered poses | pose-noise augmentation) x 3 seeds
HEADS = 'late_nat_b1,late_n_s1,late_n_s2,concat_n,concat_n_s1,concat_n_s2,win3_n,win3_n_s1,win3_n_s2,win5_n,win5_n_s1,win5_n_s2,late_a,late_a_s1,late_a_s2,concat_a,concat_a_s1,concat_a_s2,win3_a,win3_a_s1,win3_a_s2,win5_a,win5_a_s1,win5_a_s2'   #@param {type:"string"}
ZONES = 't1z5b,t1z8,t3z4,t4z1,t5b1,t5b5'         #@param {type:"string"}
SWEEPS = 'trans_shared,rot_shared'               #@param {type:"string"}
LEVELS_T = '0,0.02,0.05,0.1,0.25,0.5'            #@param {type:"string"}
LEVELS_R = '0,0.05,0.1,0.25,0.5,1'               #@param {type:"string"}
DRAWS = 10                                       #@param {type:"integer"}
MODE = 'exact'                                   #@param ["exact", "fast"]
SEED = 0                                         #@param {type:"integer"}
REFERENCE_RUN = 'exact_v1'                       #@param {type:"string"}
RUN2D = 'upernet_convnext_t_v2'                  #@param {type:"string"}
SHUTDOWN_WHEN_DONE = False                       #@param {type:"boolean"}
CACHE_DRIVE = f'{DRIVE}/cache'
CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN2D}'
ASSOC_DIR = f'{DRIVE}/fusion/assoc_{RUN2D}'
SUBSET_DIR = f'{DRIVE}/protocol/subsets_{RUN2D}'
OUT = f'{DRIVE}/protocol/{TAG}'
os.makedirs(OUT, exist_ok=True)
zones = [z.strip() for z in ZONES.split(',') if z.strip()]
heads = {}
for name in [h.strip() for h in HEADS.split(',') if h.strip()]:
    path = f'{DRIVE}/fusion/{name}/best.pth'
    assert os.path.exists(path), f"no checkpoint for head {name}: {path}"
    heads[name] = path
sweeps = [x.strip() for x in SWEEPS.split(',') if x.strip()]
assert sweeps and all(sw in ('trans_shared', 'rot_shared') for sw in sweeps), f"SWEEPS of this notebook: trans_shared, rot_shared (got {SWEEPS})"
levels = {'trans_shared': [float(x) for x in LEVELS_T.split(',')], 'rot_shared': [float(x) for x in LEVELS_R.split(',')]}
assert all(levels[sw][0] == 0.0 for sw in sweeps), "every level list starts with 0 (the delivered poses)"
for z in zones:
    assert os.path.exists(f'{CACHE_DRIVE}/{z}/points.npz'), f"no LiDAR cache on Drive for {z}"
    assert os.path.exists(f'{SUBSET_DIR}/{z}/subset.npz'), f"no feature bank on Drive for {z}: {SUBSET_DIR}/{z}/subset.npz (colab_heads_lab writes them)"
if not os.path.exists(f'{DRIVE}/protocol/{REFERENCE_RUN}/summary.json'):
    print(f"WARNING: no {DRIVE}/protocol/{REFERENCE_RUN}/summary.json - cell 7 cannot compare with the i.i.d. run")
done = []
if os.path.exists(f'{OUT}/results.json'):
    import json
    done = sorted({r['zone'] for r in json.load(open(f'{OUT}/results.json'))['runs']})
EVALS_PER_ZONE = sum(1 + (len(levels[sw]) - 1) * DRAWS for sw in sweeps)
n_models = len(heads) + 2
import numpy as np
# seconds per evaluation, measured on an A100: exact mode 6 models ~6 s (two-zone check), 26 models 13.7 s (shared_v1), 37 models
# 14.4 s (exact_v1) - interpolated, flat beyond the measured range; fast mode from fast_v1 and the heads labs
sec = float(np.interp(n_models, [6, 26, 37], [6.0, 13.7, 14.4])) if MODE == 'exact' else (0.4 + 0.1 * n_models)
todo = [z for z in zones if z not in done]
print(f"{n_models} models: LiDAR-only, ImageVote and {len(heads)} heads")
print("levels:", {sw: levels[sw] for sw in sweeps}, f"| {DRAWS} draws, {MODE} mode, seed {SEED}  ->  {OUT}")
print(f"{len(done)}/{len(zones)} zones already on Drive: {done}")
print(f"to do: {len(todo)} zones x {EVALS_PER_ZONE} evaluations = {len(todo) * EVALS_PER_ZONE}; "
      f"estimate {len(todo) * EVALS_PER_ZONE * sec / 3600:.1f} h at about {sec:.1f} s per evaluation (interpolated between measured runs)")


In [ ]:
#@title 3. Environment (restore tarball from Drive)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    assert os.path.exists(TAR), "no env tarball on Drive: run colab_train_litept cell 3 once"
    subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
PY = f'{ENV}/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pillow', 'matplotlib'], check=True)
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code + LitePT
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
assert 'trans_shared' in open('/content/code/gridnet/train/protocol.py').read() and os.path.exists('/content/code/scripts/compare_shared.py'), \
    "gridnet_code.zip on Drive is older than update41: pack and upload it again"
if not os.path.exists('/content/LitePT'):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/prs-eth/LitePT.git", "/content/LitePT"], check=True)
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/LitePT'); sys.path.insert(0,'/content/code'); "
                    "from litept.model import LitePT; import gridnet.train.protocol as P; print('imports ok', sorted(P.SHARED_SWEEPS))"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])
assert 'imports ok' in r.stdout


In [ ]:
#@title 5. Data -> local disk: LiDAR caches, feature banks, 2D caches (zone tars), camera files of ZONES
import os, sys, shutil, subprocess, time
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, '/content/code')
from gridnet.train.maps2d import cached_image_ids, zone_tar_manifest, extract_zone_tar, write_compact
DATA = '/content/data'; os.makedirs(DATA, exist_ok=True)
t0 = time.time()
for z in zones:
    for src, dst in ((f'{CACHE_DRIVE}/{z}/points.npz', f'/content/cache/{z}/points.npz'), (f'{SUBSET_DIR}/{z}/subset.npz', f'/content/banks/{z}/subset.npz')):
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst)
print(f"LiDAR caches + feature banks of {len(zones)} zones ({(time.time() - t0) / 60:.1f} min)")
CACHE2D = '/content/cache2d'; t0 = time.time(); slow = []
for z in zones:
    if zone_tar_manifest(CACHE2D_DRIVE, z, wait=30) is not None:   # one big sequential read: fast
        n = extract_zone_tar(CACHE2D_DRIVE, z, CACHE2D); how = 'tar'
    else:                                                             # no tar yet: parallel per-file copy (slow)
        ids = cached_image_ids(CACHE2D_DRIVE, z)
        with ThreadPoolExecutor(16) as ex:
            list(ex.map(lambda i: write_compact(CACHE2D_DRIVE, z, i, CACHE2D), ids))
        n = len(cached_image_ids(CACHE2D, z)); how = 'files'; slow.append(z)
    print(f"  {z}: {n} images from {how}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
if slow: print("zones without a tar (slow path) - run colab_compact_cache2d once for:", ','.join(slow))
code = ("import sys; sys.path.insert(0,'/content/code'); from gridnet.train.data2d import fetch_zones; "
        f"fetch_zones({zones!r}, '{DATA}', what=('pose',))")
subprocess.run([PY, '-c', code], check=True); print("camera files ready")
print("associations present:", [z for z in zones if os.path.exists(f'{ASSOC_DIR}/{z}/assoc.npz')], "(missing ones are computed on first use)")
print(subprocess.run(['df', '-h', '--output=avail', '/content'], capture_output=True, text=True).stdout.split()[-1], "free on the local disk")


In [ ]:
#@title 6. Protocol with shared pose errors -> GridNet/protocol/<TAG>/ (resumes per zone)
import subprocess, time, os
cmd = [PY, '/content/code/scripts/run_protocol.py', '--cache', '/content/cache', '--cache2d', CACHE2D, '--data', DATA,
       '--assoc-dir', ASSOC_DIR, '--out', OUT, '--subset-dir', '/content/banks', '--litept', '/content/LitePT',
       '--zones', ','.join(zones), '--sweeps', ','.join(sweeps), '--draws', str(DRAWS), '--mode', MODE, '--seed', str(SEED),
       '--levels-trans-shared', ','.join(f'{x:g}' for x in levels['trans_shared']), '--levels-rot-shared', ','.join(f'{x:g}' for x in levels['rot_shared'])]
cmd += ['--heads'] + [f'{n}={p}' for n, p in heads.items()]
env = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True', MPLBACKEND='Agg')
n_total = max(1, len(todo) * EVALS_PER_ZONE)
t0 = time.time(); tail = []; k = 0
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
try:
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if line.startswith('[protocol]') and ' draw ' in line:          # one line per evaluation: every 25th is shown
            k += 1
            if k % 25 == 0 or k == 1:
                el = (time.time() - t0) / 60
                print(f"   {el:5.1f} min  {k}/{n_total} evaluations, about {el / k * (n_total - k):.0f} min left  |  {line[11:150].strip()} ...", flush=True)
        elif line.startswith('[protocol]'): print(f"   {(time.time() - t0) / 60:5.1f} min  {line}", end='', flush=True)
        elif 'Error' in line or 'Traceback' in line or line.startswith('  File'): print(line, end='')
except KeyboardInterrupt:
    p.terminate(); p.wait(); print("stopped by user; finished zones are on Drive -> run this cell again to continue"); raise
p.wait(); print(f"exit {p.returncode} after {(time.time() - t0) / 60:.1f} min")
if p.returncode != 0:
    print(''.join(l for l in tail if not l.startswith('[protocol]'))[-3000:])
    print("-> if it STOPPED on 'other settings': this TAG holds a run with other heads / sweeps / levels / draws / mode - give TAG a new name in cell 2, re-run cell 2 and this cell")
assert p.returncode == 0 and os.path.exists(f'{OUT}/summary.json'), "the run did not finish: see the lines above"


In [ ]:
#@title 7. Shared against independent errors (run REFERENCE_RUN), table + figure
import subprocess, os
from IPython.display import Image, display
env = dict(os.environ, PYTHONPATH='/content/code', MPLBACKEND='Agg')
ref = f'{DRIVE}/protocol/{REFERENCE_RUN}'
if REFERENCE_RUN and os.path.exists(f'{ref}/summary.json'):
    r = subprocess.run([PY, '/content/code/scripts/compare_shared.py', '--shared', OUT, '--iid', ref, '--names', f'{TAG},{REFERENCE_RUN}',
                        '--out', f'{OUT}/compare_{REFERENCE_RUN}.json'], capture_output=True, text=True, env=env)
    print(r.stdout[-30000:], r.stderr[-1500:])
    if r.returncode == 0: open(f'{OUT}/compare_{REFERENCE_RUN}.txt', 'w').write(r.stdout)
else:
    print(f"no reference run at {ref}: comparison skipped")
one_seed = [m for m in ('lidar', 'imagevote', 'late_nat_b1', 'concat_n', 'win3_n', 'win5_n', 'late_a', 'concat_a', 'win3_a', 'win5_a') if m in heads or m in ('lidar', 'imagevote')]
fig = f'{OUT}/protocol_{TAG}.png'
r = subprocess.run([PY, '/content/code/scripts/plot_protocol.py', '--summary', f'{OUT}/summary.json', '--out', fig, '--tables', 'compact',
                    '--models', ','.join(one_seed), '--title', f'{TAG}: shared pose errors, {MODE} mode (one seed per head type)'], capture_output=True, text=True, env=env)
print(r.stdout[-6000:], r.stderr[-1500:])
if os.path.exists(fig): display(Image(fig, width=1000))
print("files on Drive:", sorted(os.listdir(OUT)))


In [ ]:
#@title 8. (optional) Shut the runtime down once everything above has finished - saves compute units on unattended runs
if SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    print("flushing pending writes to Drive ..."); drive.flush_and_unmount()
    print("results are on Drive; shutting the runtime down"); runtime.unassign()
else:
    print("SHUTDOWN_WHEN_DONE is off -> remember to Disconnect and delete runtime")


## หมายเหตุ
- error ร่วมหนึ่งตัวต่อ (โซน, draw): seed ขึ้นกับชื่อโซนด้วย แต่ละโซนของ draw เดียวกันจึงได้ error คนละตัว (6 โซน × 10 draws = 60 ตัวต่อระดับ)
  ค่าเฉลี่ยของ draw จึงแกว่งกว่า sweep i.i.d. มาก (ที่นั่นหนึ่ง draw เฉลี่ยกล้องเป็นร้อยตัวอยู่แล้ว) — ตารางของ cell 7 จึงพิมพ์ ± ของ draws ไว้ด้วย
- ระดับ 0 (pose ที่ให้มา) ต้องได้ตัวเลขเท่ากับ run `exact_v1` ทุกโมเดล (head เดิม จุดเดิม) — บรรทัด check ของ cell 7 ต้องเป็น 0.000
- ค่า i.i.d. ที่ใช้เทียบมาจาก run `exact_v1` (ไม่รันซ้ำที่นี่): โมเดลเดียวกัน, subset เดียวกัน, โหมด exact เหมือนกัน
- `trans_shared` = เลื่อนกล้องทุกตัวเท่ากันในพิกัดโลก (เทียบเท่ากับเลื่อน point cloud ทั้งก้อนไปทางตรงข้าม); `rot_shared` = หมุนในพิกัดของกล้องแต่ละตัว
  ด้วยมุมเดียวกัน (ทุกภาพเลื่อนไปทางเดียวกันในภาพ)
- จบแล้ว **Disconnect and delete runtime**